# Bronze Ingestion — TfL Line Status

Ingest the current London Underground line status from the TfL Unified API.

This notebook:

1. Loads source configuration.
2. Calls the TfL API through the reusable API client.
3. Validates the response.
4. Lands the original JSON response in a Unity Catalog Volume.
5. Appends the response and ingestion metadata to a Bronze Delta table.

**Source:** TfL Unified API  
**Target:** `workspace.urbanpulse_bronze.tfl_line_status`

Bronze data is retained with minimal transformation so that the original source response remains recoverable.

## 1. Initialise project paths

Databricks executes this notebook from the `notebooks/01_bronze` directory.

Add the repository `src` directory to the Python path so that the shared `urbanpulse` package can be imported directly from the Git folder.

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

## 2. Import project components

The notebook contains orchestration only.

Reusable HTTP, landing, configuration, and Delta-writing logic is maintained under `src/urbanpulse`.

In [0]:
import uuid

from urbanpulse.ingestion.api_clients import ApiClient
from urbanpulse.ingestion.bronze import write_raw_bronze
from urbanpulse.ingestion.landing import land_json
from urbanpulse.utils.config import load_yaml

## 3. Load source configuration

API URLs and source-specific settings are stored in `conf/sources.yml` rather than being hard-coded throughout the ingestion logic.

In [0]:
CONFIG_PATH = PROJECT_ROOT / "conf" / "sources.yml"

config = load_yaml(str(CONFIG_PATH))

tfl_config = config["tfl"]
line_status_config = tfl_config["line_status"]

BASE_URL = tfl_config["base_url"]
ENDPOINT = line_status_config["endpoint"]

LANDING_PATH = (
    "/Volumes/workspace/"
    "urbanpulse_meta/"
    "landing"
)

BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_line_status"
)

print(f"Source: {BASE_URL}{ENDPOINT}")
print(f"Target: {BRONZE_TABLE}")

## 4. Request current TfL line status

Call the TfL Unified API using the shared API client.

The client provides common timeout, retry, and HTTP error handling that can be reused by other UrbanPulse ingestion pipelines.

In [0]:
client = ApiClient(
    base_url=BASE_URL
)

payload, status_code = client.get(
    endpoint=ENDPOINT
)

request_id = str(uuid.uuid4())

print(f"HTTP status: {status_code}")
print(f"Lines returned: {len(payload)}")
print(f"Request ID: {request_id}")

## 5. Validate the source response

Perform basic contract checks before persisting the response.

Detailed field-level data-quality validation will be introduced in the Silver layer.

In [0]:
if status_code != 200:
    raise RuntimeError(
        f"TfL returned HTTP {status_code}"
    )

if not isinstance(payload, list):
    raise TypeError(
        "Expected TfL response to be a list"
    )

if not payload:
    raise ValueError(
        "TfL returned an empty response"
    )

required_fields = {
    "id",
    "name",
    "modeName",
    "lineStatuses",
}

for line in payload:
    missing_fields = (
        required_fields - set(line.keys())
    )

    if missing_fields:
        raise ValueError(
            f"TfL response is missing fields: "
            f"{sorted(missing_fields)}"
        )

print(
    f"Source validation passed for "
    f"{len(payload)} lines."
)

## 6. Land the original JSON response

Persist the unmodified API response in the Unity Catalog landing Volume.

This provides a recoverable source copy for replay, debugging, and schema-change investigation.

In [0]:
landing_file = land_json(
    payload=payload,
    base_path=LANDING_PATH,
    source="tfl",
    dataset="line_status",
    request_id=request_id,
)

print(f"Raw file landed: {landing_file}")

## 7. Append to the Bronze Delta table

Store the source payload together with ingestion metadata.

Each execution represents a new operational snapshot and is therefore appended rather than overwritten.

In [0]:
write_raw_bronze(
    spark=spark,
    payload=payload,
    request_id=request_id,
    source="tfl",
    dataset="line_status",
    source_endpoint=ENDPOINT,
    http_status=status_code,
    table_name=BRONZE_TABLE,
)

print(
    f"Bronze ingestion completed: "
    f"{request_id}"
)

## 8. Verify the landed source file

Confirm that the raw JSON file created during this execution exists in the landing Volume.

In [0]:
landing_parent = str(
    Path(landing_file).parent
)

display(
    dbutils.fs.ls(
        landing_parent
    )
)

## 9. Verify the Bronze record

Confirm that the current request was persisted successfully and inspect its ingestion metadata.

In [0]:
%sql
SELECT
    request_id,
    source,
    dataset,
    source_endpoint,
    ingested_at,
    ingestion_date,
    http_status,
    LENGTH(payload) AS payload_size
FROM workspace.urbanpulse_bronze.tfl_line_status
ORDER BY ingested_at DESC;

## 10. Bronze ingestion summary

Each successful execution produces:

`TfL API → raw JSON landing file → Bronze Delta snapshot`

Repeated executions build a historical record of TfL line-status observations.

Business transformations and nested JSON parsing are intentionally deferred to the Silver layer.

In [0]:
%sql
SELECT
    COUNT(*) AS total_snapshots,
    MIN(ingested_at) AS first_ingestion,
    MAX(ingested_at) AS latest_ingestion
FROM workspace.urbanpulse_bronze.tfl_line_status;